In [ ]:
import functools
import time


def timing_decorator(func):
    @functools.wraps(func)
    def wrapper(*args, **kwargs):
        start_time = time.time()
        result = func(*args, **kwargs)
        end_time = time.time()
        duration = end_time - start_time
        # print(f"[TIMER] {func.__name__} took {duration:.4f} seconds")
        # return result

    return wrapper

In [5]:
%run Include.ipynb
%run Net.ipynb
%run Data.ipynb
%run Topo_treatment.ipynb
%run Viewer.ipynb

import os
from datetime import datetime
from pathlib import Path

import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import pandas as pd
from tqdm.notebook import tqdm

# Added for reproducibility
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

from itertools import chain, repeat


def sticky_iterator(seq):
    return chain(seq, repeat(seq[-1]))


class GAN(object):
    def __init__(self, general, adv_params, G_arch, D_arch):
        lr = general["learning_rate"]
        beta1 = general["beta1"]
        beta2 = general["beta2"]
        loss_mode = general["loss"]
        reduction = general["reduction"]

        self.N_critic = adv_params["wgangp"]["N_CRITIC"]

        self.starting_topo_weight = 1e-7
        self.warmup_steps = (
            adv_params["Topo_edge"]["N_warmup_T_weight_EPOCHS"] // self.N_critic
        )
        self.warmup_topo_weight_schedule = sticky_iterator(
            np.linspace(self.starting_topo_weight, 1, self.warmup_steps)
        )

        cudnn.benchmark = FLAGS.cudnn_benchmark
        gpu_num = FLAGS.gpu_num
        self.device = torch.device(
            "cuda:0" if torch.cuda.is_available() and FLAGS.gpu_enable else "cpu"
        )
        #         torch.manual_seed(random.randint(1, 10000))
        # added for reproducibility
        seed = 42
        torch.manual_seed(42)
        np.random.seed(seed)
        random.seed(seed)

        self.inputG_dims, G_layers = Net.parse_layers(G_arch)
        self.inputD_dims, D_layers = Net.parse_layers(D_arch)
        self.netG = Network_template(gpu_num, G_layers).to(self.device)
        Net.init_weights(self.netG, "normal")
        self.netD = Network_template(gpu_num, D_layers).to(self.device)
        Net.init_weights(self.netD, "normal")

        self.et = Edges_(adv_params, debug=False)
        self.criterion = GANLoss(loss_mode, reduction).to(self.device)
        self.criterionT = GANLoss("vanilla_topo", "sum").to(self.device)
        self.optimizerD = optim.Adam(
            self.netD.parameters(), lr=lr, betas=(beta1, beta2)
        )
        # self.optimizerD = optim.Adam(filter(lambda p: p.requires_grad, self.netD.parameters()), lr=lr, betas=(beta1,beta2))
        self.optimizerG = optim.Adam(
            self.netG.parameters(), lr=lr, betas=(beta1, beta2)
        )

    def __call__(self, noise: torch.Tensor) -> torch.Tensor:
        """
        Make the GAN instance callable for generating images, with a shape assertion.

        Args:
            noise (Tensor): Must be of shape [batch_size, nz, 1, 1].
        Returns:
            Tensor: Generated images.
        """
        # Assert correct noise shape
        assert (
            noise.dim() == 4 and list(noise.shape[1:]) == self.inputG_dims
        ), f"Expected noise shape [batch_size, {self.inputG_dims}], got {list(noise.shape)}"
        return self.netG(noise)

    def save_noise_(self, shape, name):
        """
        shape: shape of the noise, usually it is [batch_size, 128, 1, 1]
        name: should be like 128_128_1_1_0.dat
        all noise are saved under D:/Data/fixed_z/
        """
        z_ = torch.randn(shape, device=self.device)
        FileIO.write_binary(
            "D:/Data/fixed_z/" + name, z_.cpu().numpy().flatten(), list(z_.shape), "f"
        )

    def sample_(self, shape):
        z_ = torch.randn(shape, device=self.device)
        return self.netG(z_)

    def sample_save_(self, name, shape, direc, scalor, offset):
        """
        shape: shape of the noise, usually it is [batch_size, 128, 1, 1]
        name: should be like 128_128_1_1_0.dat
        all noise are saved under D:/Data/fixed_z/
        """
        Path(direc).mkdir(parents=True, exist_ok=True)
        if FLAGS.continue_model:
            self.netG.load_state_dict(
                torch.load("%s/netG_step_%d.pth" % (FLAGS.model_save, FLAGS.model_step))
            )
            self.netD.load_state_dict(
                torch.load("%s/netD_step_%d.pth" % (FLAGS.model_save, FLAGS.model_step))
            )
            print("Models loaded at step %d" % FLAGS.model_step)

        i = 0
        interval = 1000
        z_ = FileIO.read_binary("D:/Data/fixed_z/" + name, shape, "f")
        while True:
            si = i
            se = np.min((si + interval, z_.shape[0]))
            z_sub_ = z_[si:se, :]
            z_sub_ = torch.from_numpy(z_sub_).to(self.device)
            samples = self.netG(z_sub_)
            FileIO.save_image_batch(
                samples.detach().cpu().numpy(), direc, "gen", scalor, si
            )
            i = se
            if i >= z_.shape[0]:
                break

    #     @timing_decorator
    def D_iteration(self, Dreal_device, Dfake_device):
        self.netD.zero_grad()
        errD = self.criterion(["D", self.netD, self.device, Dreal_device, Dfake_device])
        errD.backward()
        self.optimizerD.step()
        return errD.item()

    @timing_decorator
    def G_iteration(self, Dfake_device, withTopo, epoch_index: int, batch_index: int):
        # Dfake_device - сгенерированные фото
        self.netG.zero_grad()
        errG = self.criterion(["G", self.netD, Dfake_device])
        errG.backward(retain_graph=withTopo)

        if withTopo:
            tp_wgt = self.et.return_tp_weight()
            # переслать фото с видеокарточки на компьютер и применить функцию fix_with_topo
            fake_fix, mean_wasdis = self.et.fix_with_topo(
                gen=Dfake_device.detach().cpu().numpy(),
                dim=self.et.return_target_dim(),
                value=-1.0,
                wassertein_dist=1.0,
                blind=self.et.blind(),
                epoch_index=epoch_index,
                batch_index=batch_index,
            )
            fake_fix = torch.from_numpy(fake_fix).to(self.device)
            schedule_scaling = next(self.warmup_topo_weight_schedule)
            tp_wgt_with_scaling = tp_wgt * schedule_scaling
            print(
                f"Topo Weight with scheduler: {tp_wgt_with_scaling:.5f} ({tp_wgt:.3f} * {schedule_scaling})"
            )
            errT = self.criterionT([Dfake_device, fake_fix]) * tp_wgt_with_scaling
            errT.backward()
            self.optimizerG.step()
            return [errG.item(), errT.item(), mean_wasdis]
        self.optimizerG.step()

        return [errG.item(), 0.0, 0.0]

    def train(self, data_params, withTopo, add_topo_at_epoch: int = -1):
        self._init_training_state(data_params, withTopo)
        dataloader = self._get_dataloader(data_params)
        tqdm_epochs_bar = tqdm(
            range(data_params["epochs"]),
            desc="Training...",
            smoothing=0.1,  # Strong smoothing
            mininterval=0.5,  # Update bar max twice per second
            maxinterval=1.0,  # Force refresh at least once per second
            total=data_params["epochs"],
        )
        self.data_params = data_params

        for epoch in tqdm_epochs_bar:
            use_topo = withTopo and epoch >= add_topo_at_epoch
            self._train_epoch(dataloader, epoch, use_topo)

        self.log_file.close()
        print("Training complete.")

    def _init_training_state(self, data_params, withTopo):
        self.step = 0
        self.g_lrec = []
        self.d_lrec = []
        self.log_file = open(FLAGS.log_path, "a")
        self.fixed_z_ = torch.randn(
            [data_params["batch_size"], 128, 1, 1], device=self.device
        )

        if withTopo:
            self.et.load_pd_pool(FLAGS.pds_path, "dat", 1.0, data_params["batch_size"])

        if FLAGS.continue_model:
            self._load_models(FLAGS.model_step)
            self.step = FLAGS.model_step + 1

        self._init_csv_log()

    def _load_models(self, step):
        self.netG.load_state_dict(
            torch.load(f"{FLAGS.model_save}/netG_step_{step}.pth")
        )
        self.netD.load_state_dict(
            torch.load(f"{FLAGS.model_save}/netD_step_{step}.pth")
        )

    def _init_csv_log(self):
        self.csv_path = (
            f"C:\\TopoGAN-ECCV2020\\logs\\metric_log_{datetime.now():%Y%m%d_%H%M%S}.csv"
        )
        self.csv_log = open(self.csv_path, "w")
        self.csv_log.write(
            "step,epoch,batch_idx,D_loss,G_loss,T_loss,Wasserstein_dist\n"
        )

    def _get_dataloader(self, data_params):
        return Data_fetcher.fetch_dataset(
            FLAGS.dataset,
            data_params["batch_size"],
            data_params["batch_workers"],
            data_params["shuffle"],
            data_params["drop_last"],
            0.5,
        )

    def _train_epoch(self, dataloader, epoch, use_topo):
        for i, data in enumerate(dataloader):
            self._train_batch(data, epoch, i, len(dataloader), use_topo)

    def _train_batch(self, data, epoch, batch_idx, total_batches, use_topo):
        real_images = data["image"].to(self.device)
        batch_size = real_images.size(0)

        # Discriminator
        fake_images = self.sample_([batch_size] + self.inputG_dims)
        self.d_lrec.append(self.D_iteration(real_images, fake_images))

        # Generator
        if self.step % self.N_critic == 0:
            fake_images = self.sample_([batch_size] + self.inputG_dims)
            self.g_lrec.append(
                self.G_iteration(
                    fake_images,
                    withTopo=use_topo,
                    epoch_index=epoch,
                    batch_index=batch_idx,
                )
            )

        self._maybe_log(epoch, batch_idx, total_batches, use_topo)
        self._maybe_save_samples_and_models(use_topo)
        self.step += 1

    def _maybe_log(self, epoch, batch_idx, total_batches, use_topo):
        if (self.step - 1) % FLAGS.print_step != 0:
            return

        d_loss = float(np.mean(self.d_lrec))
        self.d_lrec.clear()

        try:
            g_array = np.asarray(self.g_lrec)
            self.g_lrec.clear()
            g_loss = float(np.mean(g_array[:, 0]))

            if use_topo:
                t_loss = float(np.mean(g_array[:, 1]))
                w_dist = float(np.mean(g_array[:, 2]))
                msg = f"[{epoch}/{data_params['epochs']}][{batch_idx}/{total_batches}] D: {d_loss:.4f} G: {g_loss:.4f} T: {t_loss:.4f} W: {w_dist:.4f} Step: {self.step}"
                self.csv_log.write(
                    f"{self.step},{epoch},{batch_idx},{d_loss},{g_loss},{t_loss},{w_dist}\n"
                )
            else:
                msg = f"[{epoch}/{data_params['epochs']}][{batch_idx}/{total_batches}] D: {d_loss:.4f} G: {g_loss:.4f} Step: {self.step}"
                self.csv_log.write(
                    f"{self.step},{epoch},{batch_idx},{d_loss},{g_loss},NaN,NaN\n"
                )

            print(msg)
            self.log_file.write(msg + "\n")
            self.log_file.flush()
            self.csv_log.flush()

        except IndexError:
            raise RuntimeError(
                f"Invalid shape in g_lrec. Ensure FLAGS.print_step >= N_critic and divisible. "
                f"Got print_step={FLAGS.print_step}, N_critic={self.N_critic}"
            )

    def _maybe_save_samples_and_models(self, use_topo):
        if self.step % FLAGS.save_step != 0:
            return

        print("Saving image and models...")

        with torch.no_grad():
            gen = self.netG(self.fixed_z_).cpu()
            vutils.save_image(
                gen, f"{FLAGS.image_save}/gen_step_{self.step}.png", normalize=True
            )

            if use_topo:
                fix_topo, _ = self.et.fix_with_topo(
                    gen.numpy(),
                    self.et.return_target_dim(),
                    -1.0,
                    1.0,
                    blind=self.et.blind(),
                )
                fix_topo = torch.from_numpy(np.expand_dims(fix_topo, 1))
                vutils.save_image(
                    fix_topo,
                    f"{FLAGS.image_save}/top_step_{self.step}.png",
                    normalize=True,
                )

        torch.save(
            self.netG.state_dict(), f"{FLAGS.model_save}/netG_step_{self.step}.pth"
        )
        torch.save(
            self.netD.state_dict(), f"{FLAGS.model_save}/netD_step_{self.step}.pth"
        )

    def plot_losses(self, date_str, save_path, logscale: dict = {}):
        """
        Overlay four losses each on its own colored y-axis.
        Adds date to title and saves the plot in vector format (.svg) using a full save path.

        Args:
            date_str (str): Date string in 'YYYY-MM-DD' format.
            save_path (str): Directory or file stem where image should be saved (without extension).
        """
        df = pd.read_csv(self.csv_path)
        steps = df["step"]

        # Parse date
        date_obj = datetime.strptime(date_str, "%Y-%m-%d")
        short_date = date_obj.strftime("%m-%d")  # e.g. '6 May'
        long_date = date_obj.strftime("%Y-%m-%d")  # e.g. '2025-05-06'

        fig, host = plt.subplots(figsize=(12, 6))
        fig.subplots_adjust(right=0.75)

        # Additional axes
        par1 = host.twinx()
        par2 = host.twinx()
        par3 = host.twinx()
        par2.spines["right"].set_position(("axes", 1.15))
        par3.spines["right"].set_position(("axes", 1.30))
        for p in [par2, par3]:
            p.spines["right"].set_visible(True)

        # Plot losses
        (p0,) = host.plot(steps, df["D_loss"], color="C0", label="D_loss")
        (p1,) = par1.plot(steps, df["G_loss"], color="C1", label="G_loss")
        (p2,) = par2.plot(steps, df["T_loss"], color="C2", label="T_loss")
        (p3,) = par3.plot(steps, df["Wasserstein_dist"], color="C3", label="W_dist")

        # Label axes
        host.set_xlabel("Training Step")
        host.set_ylabel("D_loss", color="C0")
        par1.set_ylabel("G_loss", color="C1")
        par2.set_ylabel("T_loss", color="C2")
        par3.set_ylabel("W_dist", color="C3")

        host.tick_params(axis="y", colors="C0")
        par1.tick_params(axis="y", colors="C1")
        par2.tick_params(axis="y", colors="C2")
        par3.tick_params(axis="y", colors="C3")

        # scales
        if logscale.get("d"):
            host.set_yscale("log")
        if logscale.get("g"):
            par1.set_yscale("log")
        if logscale.get("t"):
            par2.set_yscale("log")
        if logscale.get("w"):
            par3.set_yscale("log")

        # Title and legend
        lines = [p0, p1, p2, p3]
        host.legend(lines, [l.get_label() for l in lines], loc="upper right")
        host.set_title(f"Training Loss Progression ({short_date})")

        for ax in (host, par1, par2, par3):
            ax.yaxis.set_major_locator(
                mticker.MaxNLocator(nbins=5)
            )  # e.g. at most 5 ticks
            # 2) …and format each tick to 2 decimal places
            ax.yaxis.set_major_formatter(mticker.FormatStrFormatter("%.2f"))

        # Save plot in vector format (.svg)
        base_filename = f"{save_path}_loss_plot_{long_date}"
        filename = base_filename + ".svg"
        counter = 1

        # Check if file exists and increment counter until unique name is found
        while os.path.exists(filename):
            filename = f"{base_filename}_{counter}.svg"
            counter += 1

        host.grid(True)
        plt.savefig(filename, format="svg")
        plt.show()

    @classmethod
    def from_checkpoint(
        cls,
        ckpt_dir: str,
        step: int,
        general: dict,
        adv_params: dict,
        G_arch: list,
        D_arch: list,
        map_location=None,
    ):
        """
        Create a GAN instance and load weights from a given checkpoint step.

        Args:
            ckpt_dir (str): Directory where checkpoints are stored.
            step (int): Checkpoint number (matches save_step).
            general (dict): General training parameters (learning_rate, beta1, beta2, ...).
            adv_params (dict): Adversarial parameters (e.g., N_CRITIC for WGAN-GP).
            G_arch (list): Generator architecture description.
            D_arch (list): Discriminator architecture description.
            map_location: Device mapping for torch.load (default: same as training device).
        Returns:
            GAN: An initialized GAN object with loaded weights.
        """
        # Instantiate fresh GAN
        gan = cls(general, adv_params, G_arch, D_arch)

        # Determine device for loading
        if map_location is None:
            map_location = gan.device

        # Build checkpoint filenames
        netG_file = ckpt_dir / f"netG_step_{step}.pth"
        netD_file = ckpt_dir / f"netD_step_{step}.pth"

        # Load state dicts
        gan.netG.load_state_dict(torch.load(netG_file, map_location=map_location))
        gan.netD.load_state_dict(torch.load(netD_file, map_location=map_location))

        # Optionally, you could also load optimizer states (if saved)
        # gan.optimizerG.load_state_dict(torch.load(os.path.join(ckpt_dir, f"optimG_step_{step}.pth"), map_location))
        # gan.optimizerD.load_state_dict(torch.load(os.path.join(ckpt_dir, f"optimD_step_{step}.pth"), map_location))

        # Update FLAGS or internal counters if needed
        try:
            import FLAGS

            FLAGS.model_step = step
            FLAGS.continue_model = True
        except ImportError:
            pass

        print(f"Loaded GAN checkpoint at step {step} from {ckpt_dir}")

        return gan

NameError: name 'timing_decorator' is not defined